<a href="https://colab.research.google.com/github/HarnoorKaur812/Deep-Learning/blob/main/Self_PruningNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import math
import pickle
import urllib.request
import tarfile
import json

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

In [3]:
def sigmoid_grad(s: np.ndarray) -> np.ndarray:
    return s * (1.0 - s)

In [4]:
def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(0.0, x)

In [5]:
def relu_grad(x):
    return (x > 0).astype(np.float64)

In [6]:
def softmax(x):
    x_shifted = x - x.max(axis=1, keepdims=True)
    e = np.exp(x_shifted)
    return e / e.sum(axis=1, keepdims=True)

In [7]:
def cross_entropy_loss(probs, labels):
    batch = probs.shape[0]
    correct = probs[np.arange(batch), labels]
    return -np.log(np.clip(correct, 1e-12, 1.0)).mean()

In [8]:
def cross_entropy_softmax_grad(probs, labels):
    batch = probs.shape[0]
    grad = probs.copy()
    grad[np.arange(batch), labels] -= 1.0
    return grad / batch

In [9]:
class PrunableLinear:
    def __init__(self, in_features, out_features):
        bound = math.sqrt(2.0 / in_features)
        self.weight = np.random.uniform(-bound, bound, (out_features, in_features))
        self.bias = np.zeros(out_features)
        self.gate_scores = np.random.uniform(-4, -2, (out_features, in_features))

        self._input = None
        self._gates = None
        self._pruned_weights = None

        self.m_weight = np.zeros_like(self.weight)
        self.v_weight = np.zeros_like(self.weight)
        self.m_bias = np.zeros_like(self.bias)
        self.v_bias = np.zeros_like(self.bias)
        self.m_gate = np.zeros_like(self.gate_scores)
        self.v_gate = np.zeros_like(self.gate_scores)

    def forward(self, x):
        gates = sigmoid(self.gate_scores)
        pruned_weights = self.weight * gates
        out = x @ pruned_weights.T + self.bias

        self._input = x
        self._gates = gates
        self._pruned_weights = pruned_weights
        return out

    def backward(self, d_out):
       x = self._input
       gates = self._gates
       pruned_weights = self._pruned_weights

       d_bias = d_out.sum(axis=0)
       d_pruned_weights = d_out.T @ x
       d_weight = d_pruned_weights * gates
       d_gates = d_pruned_weights * self.weight
       d_gate_scores = d_gates * sigmoid_grad(gates)
       d_input = d_out @ pruned_weights

       self.d_weight = d_weight
       self.d_bias = d_bias
       self.d_gate_scores = d_gate_scores
       return d_input

    def adam_update(self, lr, beta1, beta2, eps, t):
        bc1 = 1.0 - beta1 ** t
        bc2 = 1.0 - beta2 ** t

        def step(param, grad, m, v):
            m[:] = beta1 * m + (1.0 - beta1) * grad
            v[:] = beta2 * v + (1.0 - beta2) * (grad ** 2)
            m_hat = m / bc1
            v_hat = v / bc2
            param -= lr * m_hat / (np.sqrt(v_hat) + eps)

        step(self.weight, self.d_weight, self.m_weight, self.v_weight)
        step(self.bias, self.d_bias, self.m_bias, self.v_bias)
        step(self.gate_scores, self.d_gate_scores, self.m_gate, self.v_gate)

    def get_gates(self):
        return sigmoid(self.gate_scores)

    def sparsity_loss_and_grad(self):
      gates = sigmoid(self.gate_scores)
      n = gates.size
      return float(gates.sum()) / n, sigmoid_grad(gates) / n

In [10]:
class SelfPruningNet:
    def __init__(self):
        self.fc1 = PrunableLinear(3072, 1024)
        self.fc2 = PrunableLinear(1024, 512)
        self.fc3 = PrunableLinear(512, 256)
        self.fc4 = PrunableLinear(256, 10)
        self.layers = [self.fc1, self.fc2, self.fc3, self.fc4]
        self._pre_relu = {}

    def forward(self, x):
        x = x.astype(np.float64)

        x = self.fc1.forward(x); self._pre_relu[1] = x; x = relu(x)
        x = self.fc2.forward(x); self._pre_relu[2] = x; x = relu(x)
        x = self.fc3.forward(x); self._pre_relu[3] = x; x = relu(x)
        return self.fc4.forward(x)

    def compute_loss(self, logits, labels, lam):
        probs = softmax(logits)
        ce = cross_entropy_loss(probs, labels)
        sp = sum(layer.sparsity_loss_and_grad()[0] for layer in self.layers)
        return ce + lam * sp, ce, sp, probs

    def backward(self, labels, probs, lam):
        d = cross_entropy_softmax_grad(probs, labels)
        d = self.fc4.backward(d)
        d = d * relu_grad(self._pre_relu[3]); d = self.fc3.backward(d)
        d = d * relu_grad(self._pre_relu[2]); d = self.fc2.backward(d)
        d = d * relu_grad(self._pre_relu[1]); self.fc1.backward(d)

        for layer in self.layers:
            _, sp_grad = layer.sparsity_loss_and_grad()
            layer.d_gate_scores += lam * sp_grad

    def adam_step(self, lr, beta1, beta2, eps, t):
        for layer in self.layers:
            layer.adam_update(lr, beta1, beta2, eps, t)

    def predict(self, x):
        return np.argmax(self.forward(x), axis=1)

    def sparsity_level(self, threshold = 0.05):
        total = sum(layer.get_gates().size for layer in self.layers)
        pruned = sum((layer.get_gates() < threshold).sum() for layer in self.layers)
        return pruned / total * 100

    def all_gates(self):
        return np.concatenate([layer.get_gates().ravel() for layer in self.layers])

In [11]:
CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"

def download_cifar10(root="./data"):
    path = os.path.join(root, "cifar-10-batches-py")
    if os.path.isdir(path): return path
    os.makedirs(root, exist_ok=True)
    tar_path = os.path.join(root, "cifar10.tar.gz")
    urllib.request.urlretrieve(CIFAR10_URL, tar_path)
    with tarfile.open(tar_path, "r:gz") as tar: tar.extractall(root)
    os.remove(tar_path)
    return path

def load_cifar10(root="./data"):
    ext = download_cifar10(root)

    Xs, ys = [], []
    for i in range(1, 6):
        with open(os.path.join(ext, f"data_batch_{i}"), "rb") as f:
            d = pickle.load(f, encoding="bytes")
        Xs.append(d[b"data"]); ys.extend(d[b"labels"])

    X_train = np.concatenate(Xs).astype(np.float64) / 255.0
    y_train = np.array(ys, dtype=np.int32)

    with open(os.path.join(ext, "test_batch"), "rb") as f:
        d = pickle.load(f, encoding="bytes")
    X_test = d[b"data"].astype(np.float64) / 255.0
    y_test = np.array(d[b"labels"], dtype=np.int32)

    mean = [0.4914, 0.4822, 0.4465]
    std = [0.2023, 0.1994, 0.2010]
    for c in range(3):
        sl = slice(c*1024, (c+1)*1024)
        X_train[:, sl] = (X_train[:, sl] - mean[c]) / std[c]
        X_test[:, sl] = (X_test[:, sl] - mean[c]) / std[c]

    return X_train, y_train, X_test, y_test

In [12]:
def iterate_batches(X, y, batch_size, shuffle=True):
    idx = np.arange(X.shape[0])
    if shuffle: np.random.shuffle(idx)
    for i in range(0, X.shape[0], batch_size):
        b = idx[i:i+batch_size]
        yield X[b], y[b]

def evaluate_accuracy(model, X, y, batch_size=512):
    correct = 0
    for xb, yb in iterate_batches(X, y, batch_size, False):
        correct += (model.predict(xb) == yb).sum()
    return correct / y.shape[0] * 100

def cosine_lr(lr, epoch, total):
    return lr * 0.5 * (1 + math.cos(math.pi * epoch / total))

def train_epoch(model, X, y, lam, lr, batch_size, step):
    total_loss = 0
    n = 0
    for xb, yb in iterate_batches(X, y, batch_size):
        logits = model.forward(xb)
        loss, _, _, probs = model.compute_loss(logits, yb, lam)
        model.backward(yb, probs, lam)

        step[0] += 1
        model.adam_step(lr, 0.9, 0.999, 1e-8, step[0])

        total_loss += loss
        n += 1
    return total_loss / n

In [ ]:
def main():
    LAMBDAS =[1e-6, 1e-5, 1e-4]

    X_train, y_train, X_test, y_test = load_cifar10()

    for lam in LAMBDAS:
        model = SelfPruningNet()
        step = [0]

        for epoch in range(30):
            lr = cosine_lr(1e-3, epoch, 30)
            loss=train_epoch(model, X_train, y_train, lam, lr, 256, step)
            acc = evaluate_accuracy(model, X_test, y_test)
            sparsity = model.sparsity_level()
            print(f"Epoch {epoch+1}/30 | Loss={loss:.4f} | Acc={acc:.2f}% | Sparsity={sparsity:.2f}%")

        acc = evaluate_accuracy(model, X_test, y_test)
        sparsity = model.sparsity_level()
        print(f"lam={lam} | acc={acc:.2f}% | sparsity={sparsity:.1f}%")

if __name__ == "__main__":
    main()

/tmp/ipykernel_3663/3430746227.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tar_path, "r:gz") as tar: tar.extractall(root)


Epoch 1/30 | Loss=2.0576 | Acc=28.13% | Sparsity=51.87%
Epoch 2/30 | Loss=1.8381 | Acc=35.33% | Sparsity=51.56%
Epoch 3/30 | Loss=1.7193 | Acc=39.34% | Sparsity=51.29%
Epoch 4/30 | Loss=1.6162 | Acc=42.96% | Sparsity=51.15%
Epoch 5/30 | Loss=1.5396 | Acc=45.30% | Sparsity=51.10%
Epoch 6/30 | Loss=1.4812 | Acc=47.15% | Sparsity=51.06%
Epoch 7/30 | Loss=1.4294 | Acc=48.39% | Sparsity=51.03%
Epoch 8/30 | Loss=1.3861 | Acc=49.40% | Sparsity=51.02%
Epoch 9/30 | Loss=1.3452 | Acc=50.37% | Sparsity=50.99%
Epoch 10/30 | Loss=1.3058 | Acc=50.97% | Sparsity=50.97%
Epoch 11/30 | Loss=1.2711 | Acc=51.69% | Sparsity=50.94%
Epoch 12/30 | Loss=1.2390 | Acc=51.88% | Sparsity=50.92%
Epoch 13/30 | Loss=1.2077 | Acc=52.74% | Sparsity=50.90%
Epoch 14/30 | Loss=1.1815 | Acc=52.47% | Sparsity=50.88%
Epoch 15/30 | Loss=1.1536 | Acc=52.96% | Sparsity=50.86%
Epoch 16/30 | Loss=1.1318 | Acc=53.05% | Sparsity=50.83%
Epoch 17/30 | Loss=1.1086 | Acc=53.04% | Sparsity=50.81%
Epoch 18/30 | Loss=1.0889 | Acc=53.48% |

In [ ]:
gates = model.all_gates()
sparsity = model.sparsity_level()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gates, bins=100, color="steelblue", edgecolor="none")
ax.set_xlabel("Gate Value")
ax.set_ylabel("Count")
ax.set_title(f"Gate Distribution | λ={lam} | Sparsity={sparsity:.1f}%")
plt.tight_layout()
plt.savefig("gate_dist_final.png", dpi=120)
plt.show()